# Phase 8 — Fine-Tuning: LoRA, QLoRA, and MLOps

## 1. Why Fine-Tune?
RAG handles factual knowledge retrieval. Fine-tuning handles: domain reasoning style, output format consistency, terminology, and behavior that cannot be prompted reliably.

In [ ]:
import sys

sys.path.insert(0, "../projects/phase8-openshift")
from qlora_finetune import FinetuneConfig, AXOLOTL_CONFIG_EXAMPLE

config = FinetuneConfig()
print("Default FinetuneConfig:")
print(config.model_dump_json(indent=2))

## 2. LoRA: Low-Rank Adaptation
Instead of updating all weights, LoRA adds two small trainable matrices A (r x d) and B (d x r). Memory cost: O(r * d) instead of O(d^2).

In [ ]:
def lora_param_count(d_model=4096, r=16, num_layers=32, modules_per_layer=4):
    original = d_model * d_model * num_layers * modules_per_layer
    lora = 2 * r * d_model * num_layers * modules_per_layer
    return original, lora, lora / original


orig, lora, ratio = lora_param_count()
print(f"Original params: {orig:,}")
print(f"LoRA params (r=16): {lora:,}")
print(f"LoRA is {ratio:.1%} of original — {1/ratio:.0f}x smaller")

## 3. QLoRA: 4-bit Quantized LoRA
Quantize base model to 4-bit (NF4) to drastically reduce VRAM. Keep adapter computation in bf16.

In [ ]:
model_sizes = {
    "7B FP16": 14,
    "7B 4-bit": 4.5,
    "13B FP16": 26,
    "13B 4-bit": 8,
    "70B FP16": 140,
    "70B 4-bit": 40,
}
print("Model VRAM requirements (GB):")
for name, gb in model_sizes.items():
    note = "<-- single consumer GPU" if gb <= 24 else ""
    print(f"  {name:15s}: {gb:5.1f} GB {note}")

## 4. Axolotl Config

In [ ]:
print(AXOLOTL_CONFIG_EXAMPLE)

## 5. MLflow Experiment Tracking

In [ ]:
from mlflow_tracking import ExperimentConfig, log_finetuning_run
from capstone import ExperimentTracker

tracker = ExperimentTracker("qlora-llama3")
for run_name, lr, loss in [
    ("run-lr-1e-4", 1e-4, 0.42),
    ("run-lr-2e-4", 2e-4, 0.38),
    ("run-lr-5e-4", 5e-4, 0.45),
]:
    mv = tracker.track_run(run_name, {"learning_rate": lr}, {"eval_loss": loss})
    print(f"{run_name}: eval_loss={loss}")

best = tracker.best_run()
print(f"\nBest: {best.run_id} (loss={best.metrics['eval_loss']})")
print(f"Quality gate passed: {tracker.quality_gate(best)}")

## Key Takeaways
- LoRA adds ~1% of original parameter count — most of the quality, fraction of the memory
- QLoRA enables 7B models on a single 8GB GPU; 13B on 12GB
- Axolotl YAML replaces ~200 lines of TRL boilerplate for standard SFT/DPO jobs
- Track every run in MLflow; use quality gates before promoting to production
- The RAFT technique (Phase 9) applies QLoRA fine-tuning specifically for RAG reasoning